# 02 — Data Cleaning
## Bluestock MF Analytics — Day 2
**Tasks:** Clean nav_history, investor_transactions, scheme_performance + 7 remaining datasets


## Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

ROOT      = Path.cwd().parent
RAW       = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

def save(df, name):
    path = PROCESSED / f"{name}_clean.csv"
    df.to_csv(path, index=False)
    print(f"  ✔ Saved → {path.name}  ({len(df):,} rows)")

print("✔ Setup complete")


## Task 1 — Clean nav_history
- Parse dates to datetime
- Sort by amfi_code + date
- Forward-fill missing NAV for holidays/weekends
- Remove duplicates
- Validate NAV > 0

In [ ]:
print("Cleaning: 02_nav_history")
df = pd.read_csv(RAW / "02_nav_history.csv", low_memory=False)
print(f"Raw shape: {df.shape}")

# Parse date
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"])

# Remove duplicates
before = len(df)
df = df.drop_duplicates(subset=["amfi_code","date"])
print(f"Duplicates removed: {before - len(df)}")

# Sort
df = df.sort_values(["amfi_code","date"]).reset_index(drop=True)

# Forward-fill per fund
frames = []
for code, grp in df.groupby("amfi_code"):
    grp = grp.set_index("date")
    full = pd.date_range(grp.index.min(), grp.index.max(), freq="D")
    grp  = grp.reindex(full)
    grp["amfi_code"] = code
    grp["nav"] = grp["nav"].ffill()
    grp.index.name = "date"
    frames.append(grp.reset_index())

df_filled = pd.concat(frames, ignore_index=True)
df_filled = df_filled[df_filled["nav"] > 0]
df_filled["amfi_code"] = df_filled["amfi_code"].astype(int)
df_filled["nav"] = df_filled["nav"].round(4)

print(f"Rows after forward-fill: {len(df_filled):,} (was {len(df):,})")
print(f"Final shape: {df_filled.shape}")
save(df_filled, "02_nav_history")


## Task 2 — Clean investor_transactions
- Standardise transaction_type
- Validate amount > 0
- Fix date formats
- Check KYC status enum values

In [ ]:
print("\nCleaning: 08_investor_transactions")
df = pd.read_csv(RAW / "08_investor_transactions.csv", low_memory=False)
print(f"Raw shape: {df.shape}")

# Parse date
df["transaction_date"] = pd.to_datetime(df["transaction_date"], errors="coerce")
df = df.dropna(subset=["transaction_date"])

# Standardise transaction_type
type_map = {"sip":"SIP","lumpsum":"Lumpsum","redemption":"Redemption",
            "switch_in":"Switch_In","switch_out":"Switch_Out","swp":"SWP","stp":"STP"}
df["transaction_type"] = df["transaction_type"].str.strip().str.lower().map(
    lambda x: type_map.get(x, x.title()))
print(f"Transaction types: {sorted(df['transaction_type'].unique())}")

# Validate amount > 0
df = df[df["amount_inr"] > 0]

# KYC status validation
valid_kyc = {"Verified","Pending","Rejected","Under Review"}
df["kyc_status"] = df["kyc_status"].str.strip().str.title()
df.loc[~df["kyc_status"].isin(valid_kyc), "kyc_status"] = "Unknown"

# Remove duplicates
df = df.drop_duplicates()
df["transaction_month"] = df["transaction_date"].dt.to_period("M").astype(str)

print(f"Final shape: {df.shape}")
save(df, "08_investor_transactions")


## Task 3 — Clean scheme_performance
- Validate all return values are numeric
- Flag anomalies
- Check expense_ratio range (0.1% – 2.5%)

In [ ]:
print("\nCleaning: 07_scheme_performance")
df = pd.read_csv(RAW / "07_scheme_performance.csv", low_memory=False)
print(f"Raw shape: {df.shape}")

metric_cols = ["return_1yr_pct","return_3yr_pct","return_5yr_pct",
               "benchmark_3yr_pct","alpha","beta","sharpe_ratio",
               "sortino_ratio","std_dev_ann_pct","max_drawdown_pct","expense_ratio_pct"]

for col in metric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        nulls = df[col].isna().sum()
        if nulls: print(f"  ⚠ {col}: {nulls} coerced to NaN")

# Expense ratio check
out_range = df[(df["expense_ratio_pct"]<0.1)|(df["expense_ratio_pct"]>2.5)]
print(f"Expense ratio out of range (0.1–2.5%): {len(out_range)}")

# max_drawdown check
pos_dd = (df["max_drawdown_pct"]>0).sum()
print(f"max_drawdown > 0 (anomaly): {pos_dd} rows — should be negative")

# Add derived column
df["alpha_vs_benchmark"] = (df["return_3yr_pct"] - df["benchmark_3yr_pct"]).round(2)
df = df.drop_duplicates(subset=["amfi_code"])

print(f"Final shape: {df.shape}")
save(df, "07_scheme_performance")


## Task 4 — Light clean remaining 7 datasets

In [ ]:
remaining = {
    "01_fund_master":         ["launch_date"],
    "03_aum_by_fund_house":   ["date"],
    "04_monthly_sip_inflows": ["month"],
    "05_category_inflows":    ["month"],
    "06_industry_folio_count":["month"],
    "09_portfolio_holdings":  ["portfolio_date"],
    "10_benchmark_indices":   ["date"],
}

for fname, date_cols in remaining.items():
    path = RAW / f"{fname}.csv"
    if not path.exists():
        print(f"  ⚠ Not found: {fname}.csv")
        continue
    df = pd.read_csv(path, low_memory=False)
    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    if "yoy_growth_pct" in df.columns:
        df["yoy_growth_pct"] = df["yoy_growth_pct"].fillna(0)
    df = df.drop_duplicates()
    save(df, fname)

print("\n✔ All 10 datasets cleaned and saved to data/processed/")
